# Bronze Layer - Raw Data Ingestion

**Purpose:** Fetch raw movie data from TMDB API and land it in bronze tables with minimal transformation.

**Data Sources:**
* TMDB `/discover/movie` endpoint (400 pages each with 3 sort orders: popularity, release_date, vote_count)
* TMDB `/movie/popular` endpoint (5 pages)
* TMDB `/movie/top_rated` endpoint (5 pages)
* TMDB `/movie/now_playing` endpoint (3 pages)
* TMDB `/movie/upcoming` endpoint (3 pages)
* TMDB `/genre/movie/list` endpoint (genre mapping)

**Output Tables:**
* `workspace.default.movies_bronze` - Raw movie data (~15-20K unique movies after deduplication)
* `workspace.default.movies_bronze_genres` - Genre ID to name mapping (19 genres)

**Execution Order:** Run this notebook first before silver and gold transformations.

In [0]:
# Standard library
import json
from datetime import datetime
from typing import Dict, List, Optional

# Third-party
import pandas as pd
import requests

In [0]:
# API Configuration
TMDB_BASE_URL = "https://api.themoviedb.org/3"
REQUESTS_TIMEOUT_SECONDS = 10
DEFAULT_PAGE_LIMIT = 5

# Table names
BRONZE_TABLE = "workspace.default.movies_bronze"
BRONZE_GENRES_TABLE = "workspace.default.movies_bronze_genres"
SILVER_TABLE = "workspace.default.movies_silver"
GOLD_BY_GENRE_TABLE = "workspace.default.movies_gold_by_genre"
GOLD_BY_DECADE_TABLE = "workspace.default.movies_gold_by_decade"

In [0]:
class MovieAPIClient:
    """Client for interacting with The Movie Database (TMDB) API.

    This class provides methods to fetch movie data including popular movies,
    detailed movie information, and genre metadata from the TMDB API.

    Attributes:
        api_key: TMDB API authentication key
        base_url: Base URL for TMDB API endpoints
        session: Requests session for connection pooling
    """

    def __init__(self, api_key: str, base_url: str = TMDB_BASE_URL):
        """Initializes the MovieAPIClient.

        Args:
            api_key: TMDB API key for authentication
            base_url: Base URL for API requests (default: TMDB_BASE_URL constant)

        Raises:
            ValueError: If api_key is empty or None
        """
        if not api_key:
            raise ValueError(
                "API key cannot be empty. Please provide a valid TMDB API key."
            )

        self.api_key = api_key
        self.base_url = base_url
        self.session = requests.Session()

    def get_popular_movies(self, pages: int = DEFAULT_PAGE_LIMIT) -> List[Dict]:
        """Fetches popular movies from TMDB API.

        Args:
            pages: Number of pages to fetch (default: DEFAULT_PAGE_LIMIT)

        Returns:
            List of movie dictionaries containing movie metadata

        Raises:
            requests.exceptions.RequestException: If API request fails
            ValueError: If pages is less than 1
        """
        if pages < 1:
            raise ValueError(f"Pages must be >= 1, got {pages}")

        all_movies = []

        for page in range(1, pages + 1):
            try:
                url = f"{self.base_url}/movie/popular"
                params = {"api_key": self.api_key, "page": page}

                response = self.session.get(
                    url, params=params, timeout=REQUESTS_TIMEOUT_SECONDS
                )
                response.raise_for_status()

                data = response.json()
                all_movies.extend(data.get("results", []))

                print(
                    f"Fetched page {page}/{pages}: {len(data.get('results', []))} movies"
                )

            except requests.exceptions.Timeout:
                print(f"Warning: Request timeout on page {page}. Skipping.")
                continue
            except requests.exceptions.RequestException as e:
                print(f"Error fetching page {page}: {e}")
                raise

        return all_movies

    def get_top_rated_movies(self, pages: int = DEFAULT_PAGE_LIMIT) -> List[Dict]:
        """Fetches top rated movies from TMDB API.

        Args:
            pages: Number of pages to fetch (default: DEFAULT_PAGE_LIMIT)

        Returns:
            List of movie dictionaries containing movie metadata

        Raises:
            requests.exceptions.RequestException: If API request fails
            ValueError: If pages is less than 1
        """
        if pages < 1:
            raise ValueError(f"Pages must be >= 1, got {pages}")

        all_movies = []

        for page in range(1, pages + 1):
            try:
                url = f"{self.base_url}/movie/top_rated"
                params = {"api_key": self.api_key, "page": page}

                response = self.session.get(
                    url, params=params, timeout=REQUESTS_TIMEOUT_SECONDS
                )
                response.raise_for_status()

                data = response.json()
                all_movies.extend(data.get("results", []))

                print(
                    f"Fetched top rated page {page}/{pages}: {len(data.get('results', []))} movies"
                )

            except requests.exceptions.Timeout:
                print(f"Warning: Request timeout on page {page}. Skipping.")
                continue
            except requests.exceptions.RequestException as e:
                print(f"Error fetching page {page}: {e}")
                raise

        return all_movies

    def get_now_playing_movies(self, pages: int = DEFAULT_PAGE_LIMIT) -> List[Dict]:
        """Fetches currently playing movies from TMDB API.

        Args:
            pages: Number of pages to fetch (default: DEFAULT_PAGE_LIMIT)

        Returns:
            List of movie dictionaries containing movie metadata

        Raises:
            requests.exceptions.RequestException: If API request fails
            ValueError: If pages is less than 1
        """
        if pages < 1:
            raise ValueError(f"Pages must be >= 1, got {pages}")

        all_movies = []

        for page in range(1, pages + 1):
            try:
                url = f"{self.base_url}/movie/now_playing"
                params = {"api_key": self.api_key, "page": page}

                response = self.session.get(
                    url, params=params, timeout=REQUESTS_TIMEOUT_SECONDS
                )
                response.raise_for_status()

                data = response.json()
                all_movies.extend(data.get("results", []))

                print(
                    f"Fetched now playing page {page}/{pages}: {len(data.get('results', []))} movies"
                )

            except requests.exceptions.Timeout:
                print(f"Warning: Request timeout on page {page}. Skipping.")
                continue
            except requests.exceptions.RequestException as e:
                print(f"Error fetching page {page}: {e}")
                raise

        return all_movies

    def get_upcoming_movies(self, pages: int = DEFAULT_PAGE_LIMIT) -> List[Dict]:
        """Fetches upcoming movies from TMDB API.

        Args:
            pages: Number of pages to fetch (default: DEFAULT_PAGE_LIMIT)

        Returns:
            List of movie dictionaries containing movie metadata

        Raises:
            requests.exceptions.RequestException: If API request fails
            ValueError: If pages is less than 1
        """
        if pages < 1:
            raise ValueError(f"Pages must be >= 1, got {pages}")

        all_movies = []

        for page in range(1, pages + 1):
            try:
                url = f"{self.base_url}/movie/upcoming"
                params = {"api_key": self.api_key, "page": page}

                response = self.session.get(
                    url, params=params, timeout=REQUESTS_TIMEOUT_SECONDS
                )
                response.raise_for_status()

                data = response.json()
                all_movies.extend(data.get("results", []))

                print(
                    f"Fetched upcoming page {page}/{pages}: {len(data.get('results', []))} movies"
                )

            except requests.exceptions.Timeout:
                print(f"Warning: Request timeout on page {page}. Skipping.")
                continue
            except requests.exceptions.RequestException as e:
                print(f"Error fetching page {page}: {e}")
                raise

        return all_movies

    def get_movies_by_year_range(
        self,
        start_year: int,
        end_year: int,
        pages: int = DEFAULT_PAGE_LIMIT,
        sort_by: str = "popularity.desc",
    ) -> List[Dict]:
        """Fetches movies released within a specific year range using discover endpoint.

        Args:
            start_year: Starting year (inclusive)
            end_year: Ending year (inclusive)
            pages: Number of pages to fetch (default: DEFAULT_PAGE_LIMIT)
            sort_by: Sort order for results (default: popularity.desc)
                Options: popularity.desc, popularity.asc, release_date.desc,
                release_date.asc, vote_average.desc, vote_count.desc

        Returns:
            List of movie dictionaries containing movie metadata

        Raises:
            requests.exceptions.RequestException: If API request fails
            ValueError: If years are invalid or pages is less than 1
        """
        if pages < 1:
            raise ValueError(f"Pages must be >= 1, got {pages}")
        if start_year > end_year:
            raise ValueError(
                f"Start year ({start_year}) must be <= end year ({end_year})"
            )

        all_movies = []

        for page in range(1, pages + 1):
            try:
                url = f"{self.base_url}/discover/movie"
                params = {
                    "api_key": self.api_key,
                    "page": page,
                    "primary_release_date.gte": f"{start_year}-01-01",
                    "primary_release_date.lte": f"{end_year}-12-31",
                    "sort_by": sort_by,
                }

                response = self.session.get(
                    url, params=params, timeout=REQUESTS_TIMEOUT_SECONDS
                )
                response.raise_for_status()

                data = response.json()
                results = data.get("results", [])
                all_movies.extend(results)

                print(
                    f"Fetched discover page {page}/{pages} ({start_year}-{end_year}, {sort_by}): {len(results)} movies"
                )

                if not results:
                    print(f"No more results after page {page}. Stopping.")
                    break

            except requests.exceptions.Timeout:
                print(f"Warning: Request timeout on page {page}. Skipping.")
                continue
            except requests.exceptions.RequestException as e:
                print(f"Error fetching page {page}: {e}")
                raise

        return all_movies

    def get_movie_details(self, movie_id: int) -> Optional[Dict]:
        """Fetches detailed information for a specific movie.

        Args:
            movie_id: TMDB movie ID

        Returns:
            Dictionary containing detailed movie information, or None if request fails

        Raises:
            requests.exceptions.RequestException: If API request fails critically
        """
        try:
            url = f"{self.base_url}/movie/{movie_id}"
            params = {"api_key": self.api_key}

            response = self.session.get(
                url, params=params, timeout=REQUESTS_TIMEOUT_SECONDS
            )
            response.raise_for_status()

            return response.json()

        except requests.exceptions.RequestException as e:
            print(f"Error fetching details for movie {movie_id}: {e}")
            return None

    def get_genres(self) -> List[Dict]:
        """Fetches available movie genres from TMDB.

        Returns:
            List of genre dictionaries with id and name

        Raises:
            requests.exceptions.RequestException: If API request fails
        """
        try:
            url = f"{self.base_url}/genre/movie/list"
            params = {"api_key": self.api_key}

            response = self.session.get(
                url, params=params, timeout=REQUESTS_TIMEOUT_SECONDS
            )
            response.raise_for_status()

            data = response.json()
            return data.get("genres", [])

        except requests.exceptions.RequestException as e:
            print(f"Error fetching genres: {e}")
            raise

In [0]:
def flatten_movie_data(movies: List[Dict]) -> List[Dict]:
    """Flattens nested movie data from TMDB API into a tabular structure.

    Converts nested JSON structures (like genre_ids arrays) into a format
    suitable for DataFrame creation.

    Args:
        movies: List of movie dictionaries from TMDB API

    Returns:
        List of flattened movie dictionaries with simplified structure

    Raises:
        TypeError: If movies is not a list
    """
    if not isinstance(movies, list):
        raise TypeError(f"Expected list, got {type(movies).__name__}")

    flattened = []

    for movie in movies:
        flat_movie = {
            "movie_id": movie.get("id"),
            "title": movie.get("title"),
            "original_title": movie.get("original_title"),
            "overview": movie.get("overview"),
            "release_date": movie.get("release_date"),
            "popularity": movie.get("popularity"),
            "vote_average": movie.get("vote_average"),
            "vote_count": movie.get("vote_count"),
            "genre_ids": json.dumps(movie.get("genre_ids", [])),
            "original_language": movie.get("original_language"),
            "adult": movie.get("adult"),
            "poster_path": movie.get("poster_path"),
            "backdrop_path": movie.get("backdrop_path"),
            "ingestion_timestamp": datetime.now().isoformat(),
        }
        flattened.append(flat_movie)

    return flattened

In [0]:
# Retrieve API key and initialize client
api_key = dbutils.secrets.get(scope="tmdb", key="api_key")
client = MovieAPIClient(api_key=api_key)

# Fetch movies from multiple endpoints with different sort orders for comprehensive coverage
# Strategy: Use discover endpoint with multiple sort criteria to capture diverse movie types
# - popularity.desc: Mainstream hits and blockbusters
# - release_date.desc: Recent releases regardless of popularity
# - vote_count.desc: Movies with high engagement and lots of user ratings

print("Fetching movies from TMDB API...\n")

# Primary source: Movies since 2000 with multiple sort orders
print("1. Fetching popular movies since 2000 (sorted by popularity)...")
modern_popular = client.get_movies_by_year_range(
    start_year=2000, end_year=2026, pages=400, sort_by="popularity.desc"
)
print(f"Popular movies: {len(modern_popular)}\n")

print("2. Fetching recent releases since 2000 (sorted by release date)...")
modern_recent = client.get_movies_by_year_range(
    start_year=2000, end_year=2026, pages=400, sort_by="release_date.desc"
)
print(f"Recent releases: {len(modern_recent)}\n")

print("3. Fetching highly-rated movies since 2000 (sorted by vote count)...")
modern_voted = client.get_movies_by_year_range(
    start_year=2000, end_year=2026, pages=400, sort_by="vote_count.desc"
)
print(f"Highly-rated movies: {len(modern_voted)}\n")

# Supplementary sources for additional diversity
print("4. Fetching current popular movies...")
popular_movies = client.get_popular_movies(pages=5)
print(f"Current popular: {len(popular_movies)}\n")

print("5. Fetching top rated movies...")
top_rated_movies = client.get_top_rated_movies(pages=5)
print(f"Top rated: {len(top_rated_movies)}\n")

print("6. Fetching now playing movies...")
now_playing_movies = client.get_now_playing_movies(pages=3)
print(f"Now playing: {len(now_playing_movies)}\n")

print("7. Fetching upcoming movies...")
upcoming_movies = client.get_upcoming_movies(pages=3)
print(f"Upcoming: {len(upcoming_movies)}\n")

# Combine all movie sources
all_movies = (
    modern_popular
    + modern_recent
    + modern_voted
    + popular_movies
    + top_rated_movies
    + now_playing_movies
    + upcoming_movies
)
print(f"Total movies before deduplication: {len(all_movies)}")

# Deduplicate by movie ID to remove overlaps between different sources
seen_ids = set()
unique_movies = []
for movie in all_movies:
    movie_id = movie.get("id")
    if movie_id and movie_id not in seen_ids:
        seen_ids.add(movie_id)
        unique_movies.append(movie)

print(f"Total unique movies after deduplication: {len(unique_movies)}\n")

# Flatten nested JSON structure and convert to Spark DataFrame
flattened_movies = flatten_movie_data(unique_movies)
movies_pdf = pd.DataFrame(flattened_movies)
movies_df = spark.createDataFrame(movies_pdf)

print("Sample of ingested data:")
display(movies_df.limit(5))

In [0]:
# Register DataFrame as temporary view for SQL access
movies_df.createOrReplaceTempView("movies_df")

In [0]:
# Purpose: Create bronze layer table with raw movie data from TMDB API
# Input: movies_df DataFrame from API
# Output: workspace.default.movies_bronze table

# Write DataFrame to Delta table
movies_df.write.mode("overwrite").saveAsTable(BRONZE_TABLE)

# Add table comment
spark.sql(f"""
    COMMENT ON TABLE {BRONZE_TABLE} IS
    'Bronze layer: Raw movie data from TMDB API. No transformations applied. Refreshed on-demand via API ingestion. Source: TMDB API multiple endpoints (discover, popular, top_rated, now_playing, upcoming).'
""")

# Add column comments for documentation
spark.sql(f"""
    ALTER TABLE {BRONZE_TABLE}
    ALTER COLUMN movie_id
    COMMENT 'Unique TMDB movie identifier'
""")

spark.sql(f"""
    ALTER TABLE {BRONZE_TABLE}
    ALTER COLUMN genre_ids
    COMMENT 'JSON array of genre IDs associated with this movie'
""")

spark.sql(f"""
    ALTER TABLE {BRONZE_TABLE}
    ALTER COLUMN ingestion_timestamp
    COMMENT 'Timestamp when data was ingested from API'
""")

# Display row count
total_movies = spark.table(BRONZE_TABLE).count()
print(f"Total movies in bronze table: {total_movies}")
display(spark.sql(f"SELECT COUNT(*) AS total_movies FROM {BRONZE_TABLE}"))

In [0]:
# Fetch genre mapping from TMDB API
print("Fetching genre list from TMDB API...")
genres_list = client.get_genres()
print(f"Total genres fetched: {len(genres_list)}")

# Convert to Spark DataFrame
genres_pdf = pd.DataFrame(genres_list)
genres_pdf = genres_pdf.rename(columns={"id": "genre_id", "name": "genre_name"})
genres_df = spark.createDataFrame(genres_pdf)

print("\nGenre mapping:")
display(genres_df)

In [0]:
# Register DataFrame as temporary view for SQL access
genres_df.createOrReplaceTempView("genres_df")

In [0]:
# Purpose: Create bronze layer genres lookup table
# Input: genres_df DataFrame from API
# Output: workspace.default.movies_bronze_genres table

# Write DataFrame to Delta table
genres_df.write.mode("overwrite").saveAsTable(BRONZE_GENRES_TABLE)

# Add table comment
spark.sql(f"""
    COMMENT ON TABLE {BRONZE_GENRES_TABLE} IS
    'Bronze layer: Genre ID to name mapping from TMDB API. Static reference data refreshed on-demand. Source: TMDB API /genre/movie/list endpoint.'
""")

# Add column comments for documentation
spark.sql(f"""
    ALTER TABLE {BRONZE_GENRES_TABLE}
    ALTER COLUMN genre_id
    COMMENT 'Unique TMDB genre identifier'
""")

spark.sql(f"""
    ALTER TABLE {BRONZE_GENRES_TABLE}
    ALTER COLUMN genre_name
    COMMENT 'Human-readable genre name'
""")

# Display genres sorted by name
print(f"\nGenre mapping stored in {BRONZE_GENRES_TABLE}:")
display(spark.table(BRONZE_GENRES_TABLE).orderBy("genre_name"))